COSC 301 Project
Group 4
Instacart Market Basket Analysis

**Importing Files**

In [2]:
import pandas as pd               # for data manipulation
import matplotlib.pyplot as plt   # for plotting 
import seaborn as sns             # for statistical graph
import sqlite3                     # for database connection

In [4]:
orders = pd.read_csv('../raw_data/orders.csv' )
products = pd.read_csv('../raw_data/products.csv')
order_products_prior = pd.read_csv('../raw_data/order_products__prior.csv')
aisles = pd.read_csv('../raw_data/aisles.csv')

In [4]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [5]:
order_products_prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [5]:
prior_orders = pd.merge(orders, order_products_prior, on='order_id', how='inner')
print(prior_orders.memory_usage(deep=True).sum() / (1024**3))
print(len(orders))
print(len(products))
print(len(order_products_prior))
print(len(aisles))
print(f"prior_orders: {len(prior_orders)} rows")


3.8060785699635744
3421083
49688
32434489
134
prior_orders: 32434489 rows


In [13]:
prior_orders.head()


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
5,2398795,1,prior,2,3,7,15.0,196,1,1
6,2398795,1,prior,2,3,7,15.0,10258,2,0
7,2398795,1,prior,2,3,7,15.0,12427,3,1
8,2398795,1,prior,2,3,7,15.0,13176,4,0
9,2398795,1,prior,2,3,7,15.0,26088,5,1


## Cleaning data

How to clean 
drop columns
remove na values or rows or columns
maybe check how many na values in a row or column and only remove if greater than 5 or 10% or maybe remove all rows that have an na in them


In [ ]:
#code for removing na 
prior_orders_cleaned = prior_orders.dropna()

print(f"prior_orders_cleaned: {len(prior_orders_cleaned)} rows")

prior_orders_cleaned: 30356421 rows


Research Question 1

How accurately will we recommend the top 5 products a user is most likely to purchase in their next order based on their previous orders (using the order_product_prior csv file)?

Steps to answer the question

Step 1:
create dataframe with all except last order of each user. use a loop to create dataframe for each user.

Step 2: 
Calculate top 5 products based on frequency of ordering the product that the user has order in all of their order excluding their latest order
OR
Use ML algorithm to recommend 5 products based on what user has previously ordered
OR 
BOTH

Step 3:
recommend top 5 products a user will order in the next order based on previous orders.
compute accuracy by comparing the recommended 5 products with the actual orders ordered in the last order of the user

In [6]:
#step 1

#making the connection to a created database
connection = sqlite3.connect("instacart.db")

#sending dataframe to the sql database
prior_orders.to_sql("prior_orders", connection, if_exists = "replace", index = False)

query = """
        SELECT order_id, user_id, product_id, order_number
        FROM (
            SELECT 
                order_id,
                user_id,
                product_id,
                order_number,
                MAX(order_number) OVER (PARTITION BY user_id) AS max_order_number
            FROM prior_orders 
        )
        WHERE order_number < max_order_number 
    """
    
df_without_last_orders_of_users = pd.read_sql(query, connection)